# Installation & imports

In [3]:
# 🧰 Install required Python libraries (CPU-only environment)

# Upgrade pip
%pip install --upgrade pip

# Install PyTorch for CPU
%pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cpu

# Install Hugging Face Transformers
%pip install transformers==4.51.3

# Install OpenVINO Runtime
%pip install openvino==2025.1.0

# Install visualization libraries
%pip install matplotlib==3.10.3 seaborn==0.13.2

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification, pipeline
from openvino import Core, convert_model
import time
import torch

/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model Loading using HuggingFace

In [5]:
model_id = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)


# Export the Model to ONNX Format

WARN: it will take time ..... you can take a ☕

In [ ]:
# Reduce max length to make the model graph smaller
dummy_input = tokenizer("OpenVINO is fast", return_tensors="pt", max_length=32, truncation=True)

# Ensure the model is on CPU
model.to("cpu")

torch.autograd.set_detect_anomaly(True)
torch.onnx.export(
    model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    "distilbert.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"},
    },
    opset_version=13,
    do_constant_folding=True,  # Optional, can reduce ops
)
torch.autograd.set_detect_anomaly(False)

# Convert ONNX Model to OpenVINO IR Format

In [1]:
%mo --input_model distilbert.onnx --framework onnx --output_dir openvino_model

UsageError: Line magic function `%mo` not found.


# CPU compilation

In [ ]:
ie = Core()
model = ie.read_model(model="openvino_model/distilbert.xml")
compiled_model = ie.compile_model(model=model, device_name="CPU")

# Measure Inference Time with Native PyTorch Model

In [14]:
# Prepare input
input_text = "OpenVINO makes inference really fast on Intel CPUs."
inputs = tokenizer(input_text, return_tensors="pt")

# Warm-up
for _ in range(5):
    _ = model(**inputs)

# Measure inference time
start = time.time()
with torch.no_grad():
    for _ in range(100):  # Run multiple times for averaging
        _ = model(**inputs)
end = time.time()

avg_inference_time = (end - start) / 100
print(f"PyTorch average inference time: {avg_inference_time:.4f} seconds")

PyTorch average inference time: 0.0127 seconds


# Measure Inference with OpenVino model

In [ ]:
ov_inputs = {k: v.numpy() for k, v in inputs.items()}

start_time = time.time()
output = compiled_model(ov_inputs)
end_time = time.time()

print(f"Inference time: {(end_time - start_time)*1000:.2f} ms")